In [0]:
%sql
with full_list as (
  -- Owner Document IDs
  select distinct cms1.Document_Id, ud1.BU from 
  dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_full_records_metadata as cms1
  left join dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_user_access_audit as uaa1
  on trim(uaa1.WEBI_Doc_ID)=trim(cms1.Document_Id)
  left join
  dataplatform01_central_dev_catalog_01.custom_sap_bo.ad_user_profiles as ud1
  on upper(trim(ud1.user_ID)) = upper(trim(
    case 
      when cms1.created is null or upper(trim(cms1.created)) = 'ADMINISTRATOR' then cms1.lastAuthor
      else cms1.created
    end
  ))

  Union all
  -- Viewing Document IDs
  select distinct cms1.Document_Id, ud1.BU from 
  dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_full_records_metadata as cms1
  left join dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_user_access_audit as uaa1
  on trim(uaa1.WEBI_Doc_ID)=trim(cms1.Document_Id)
  left join
  dataplatform01_central_dev_catalog_01.custom_sap_bo.ad_user_profiles as ud1
  on upper(trim(ud1.user_ID)) = upper(trim(uaa1.UserName))
),
scanned_record as (
  select *
from (
  select *,
    row_number() over (partition by document_id order by cast(ingestion_ts as timestamp) desc) as rn
  from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_document_extraction_summary
  -- where document_id='372638'
)
where rn = 1
),
inscope_list as(
  select distinct Document_Id from full_list
  where 
  Document_Id in (select distinct Document_Id from dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_metadata_cms_silver)
  and 
  Document_Id not in (select distinct Document_Id from scanned_record where document_name is not null)
), 
Audit_data as (
  select
    UA1.*,
    case
      when UA1.event_type in ('Edit', 'Deliver', 'Modify', 'Save', 'Delete', 'Create') then 'Editor_events'
      else 'Viewer_events'
    end as usage_category,
    UP1.BU
  from dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_user_access_audit as UA1
  left join dataplatform01_central_dev_catalog_01.custom_sap_bo.ad_user_profiles as UP1
  on upper(trim(UA1.UserName)) = upper(trim(UP1.user_ID))
  where upper(trim(UP1.user_ID)) in (select distinct upper(trim(user_ID)) from dataplatform01_central_dev_catalog_01.custom_sap_bo.controller_team )

),
Audit_profile as (
    SELECT
    WEBI_Doc_ID,
    COUNT(DISTINCT UserName) AS user_count,
    COUNT(DISTINCT BU) AS bu_count
    FROM Audit_data
    GROUP BY WEBI_Doc_ID
),
total_in_scope as (
  select count(distinct Document_Id) as total_cnt from (
    select distinct Document_Id from full_list
    where Document_Id in (select distinct Document_Id from dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_metadata_cms_silver)
  )
)
select 'Total extracted' as Status, 
       format_number(count(distinct Document_Id), 0) as BO_REPORTS_CNT,
       format_number(100.0 * count(distinct Document_Id) / 22494, 2) || '%' as Portion
from scanned_record, total_in_scope t
where document_name is not null
union all
select 'Total inscope pending extraction' as Status, 
       format_number(count(distinct Document_Id), 0) as BO_REPORTS_CNT,
       format_number(100.0 * count(distinct Document_Id) / 22494, 2)|| '%'  as Portion
from inscope_list, total_in_scope t
union all 
select 'Multiple BUs Controller accessed' as Status, 
       format_number(count(distinct Document_Id), 0) as BO_REPORTS_CNT,
       '100%' as Progress
from inscope_list as l1 
left join Audit_profile as a1
on trim(l1.Document_Id) = trim(a1.WEBI_Doc_ID), total_in_scope t
where a1.WEBI_Doc_ID is not null
and a1.bu_count > 1
union all
select 'Multiple Controller accessed' as Status, 
       format_number(count(distinct Document_Id), 0) as BO_REPORTS_CNT,
       '100%' as Progress
from inscope_list as l1 
left join Audit_profile as a1
on trim(l1.Document_Id) = trim(a1.WEBI_Doc_ID), total_in_scope t
where a1.WEBI_Doc_ID is not null
and a1.user_count > 1 and a1.bu_count <= 1
union all
select 'Single Controller accessed' as Status, 
       format_number(count(distinct Document_Id), 0) as BO_REPORTS_CNT,
       format_number(100.0 * (1-count(distinct Document_Id) / 5393), 2)||'%' as Portion
from inscope_list as l1 
left join Audit_profile as a1
on trim(l1.Document_Id) = trim(a1.WEBI_Doc_ID), total_in_scope t
where a1.WEBI_Doc_ID is not null
and a1.user_count = 1
union all
select 'Outside of Controller´s access' as Status, 
       format_number(count(distinct l1.Document_Id), 0) as BO_REPORTS_CNT,
       format_number(100.0 * count(distinct l1.Document_Id) / 22494, 2) || '%' as Portion
from inscope_list as l1 
left join Audit_profile as a1
on trim(l1.Document_Id) = trim(a1.WEBI_Doc_ID), total_in_scope t
where a1.WEBI_Doc_ID is null


In [0]:
%sql

select 
  to_date(ingestion_ts) as Extraction_date, 
  format_number(ceil((unix_timestamp(max(ingestion_ts)) - unix_timestamp(min(ingestion_ts))) / 1800) * 0.5, 1) as Processing_hours,
  format_number(count(distinct Document_Id),0) as Processed_reports_cnt, 
  format_number(floor(count(distinct Document_Id) / nullif(ceil((unix_timestamp(max(ingestion_ts)) - unix_timestamp(min(ingestion_ts))) / 3600, 2), 0) / 5) * 5,0) as Hourly_velocity
from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_document_extraction_summary
where ingestion_ts is not null
-- and document_name is not null
group by to_date( ingestion_ts )
order by to_date( ingestion_ts ) desc

-- select * from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_document_extraction_summary
-- where  document_name  like '%not exist%'


In [0]:
%sql
select * from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_document_extraction_summary
where to_date(ingestion_ts)=current_date()    

In [0]:
from pyspark.sql import Row

# # BU_Group="COMMERCIALS"
# bu_filter = ['COMCE-GERMANY,AUSTRIA, SWITZERL,CEE', 'COMAS-ASIA', 'GLOBA-GLOBAL','COMNA-NORTH AMERICA','COMNN-NETHERLANDS','COMON-OCEANIA','COMSE-BELGIUM, LUXEMBOURG','COMSPB-SPAIN, PORTUGAL, BRAZIL','COMUI-UK & IRELAND','FRANC-FRANCE','ITALY-ITALY','CREDIT INSURANCE' ]

# In Progress BU
bu_filter = ['COMNA-North America']

# # BU	cnt
# # FRANC-France	60
# # COMSPB-Spain, Portugal, Brazil	84
# # COMON-Oceania	216
# # COMNA-North America	351
# # COMAS-Asia	370
# # GLOBA-Global	407
# # ITALY-Italy	447
# # COMNN-Netherlands	629
# # COMUI-UK & Ireland	919
# # COMSE-Belgium, Luxembourg	1817
# # COMCE-Germany,Austria, Switzerl,CEE	1964

# # Controller BU scanned list
# bu_filter = ['FRANC-France','COMSPB-Spain, Portugal, Brazil','COMON-Oceania']


# # Completed Extraction BUs
# bu_filter = ['RISK6-RS6-CEE and ITA','GMC-GROUP MARKETING & COMMUNICATION','GRC&R-GROUP CLAIMS & RECOVERIES','FINPM-FINANCE PROGRAM MANAGEMENT','GFO-GROUP FINANCE OPERATIONS','GFC-GROUP FINANCE AND CONTROL','COFIN-CORPORATE FINANCE & TAX','Group Finance','Finance','Finance & Control','RISK1-RS1-GERMANY, CENTRAL & EAST EUROPE','RISK2-RS2-BELGIUM, LUX, FRANCE & ITA','RISK3-RS3-NLD, APAC & AIM','RISK4-RS4-NORTH EUROPE, CIS & ACS','RISK4-RS4-NORTH EUROPE, CIS & SP','RISK5-RS5-NAFTA','RISK1-RS1-DEU, AUT and CHE','RISK1-RS1-Europe, Russia & CIS','RISK2-RS2-NLD, Belux, FRA, Africa & ISR','RISK3-RS3-Asia and Oceania','RISK7-RS7-Spain, Portugal, Andorra']

bu_df = spark.createDataFrame([Row(bu=b) for b in bu_filter])
bu_df.createOrReplaceTempView("vw_bu_filter")

print(bu_filter)

In [0]:
%sql
select * from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_document_extraction_summary
where ingestion_ts = (select max(ingestion_ts) from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_document_extraction_summary)